# LegalQA v9.2 — evidence-grounded QLoRA trên official train
Đầu vào bắt buộc là train_clean.json (7.000 mẫu), cùng thư mục với public-official.json và training_quality.json. Không tìm hay dùng thay bằng train.json/train_clean_qlora.json. Runner áp dụng 47 ID loại trong training_quality.json, tự tạo prepared_train/qlora_train.json (6.953 mẫu) dưới /kaggle/working. TRAIN_SCOPE='split' giữ khoảng 400 câu grouped-dev; 'full' dùng toàn bộ mẫu hợp lệ. Giữ corpus/model/selector baseline 0.5007. Lần đầu: AUTO_RESUME_TRAINING=False, RESUME_INPUT=None, session mới. Resume lần train sạch này: AUTO_RESUME_TRAINING=True và attach recovery cùng dữ liệu.

In [ ]:
TRAIN_SCOPE = 'split'       # 'split' để calibration; 'full' cho adapter public cuối
DEV_SIZE = 400
EPOCHS = 1.0
GRAD_ACCUM = 16
MAX_LENGTH = 1536
MAX_TARGET_TOKENS = 700
MAX_EVIDENCE_CHARS = 5000
LORA_R = 8
CHECKPOINT_STEPS = 20
SAVE_TOTAL_LIMIT = 2
MAX_RUNTIME_HOURS = 11.0  # dừng an toàn trước hard limit 12h của Kaggle
RECOVERY_OUTPUT = '/kaggle/working/qlora_recovery_latest.zip'
MAX_TRAIN_SAMPLES = None    # đặt 300 để smoke; None là toàn bộ split/full
AUTO_RESUME_TRAINING = False  # lần đầu train sạch; True khi tiếp tục recovery cùng dữ liệu
RESUME_INPUT = None          # Đặt đường dẫn chính xác nếu attach nhiều training state


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
def require_kaggle_cuda():
    nvidia_smi = shutil.which('nvidia-smi')
    if nvidia_smi is None:
        raise RuntimeError('Session Kaggle chưa được gắn GPU. Vào Settings/Session options → Accelerator → GPU (T4 x2 hoặc T4 x1), restart session rồi Run All.')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError('Kaggle có nvidia-smi nhưng PyTorch không thấy CUDA. Hãy restart session sau khi bật GPU.')
    subprocess.run([nvidia_smi, '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
    print(f'CUDA OK: {torch.cuda.device_count()} GPU; pipeline dùng cuda:0')
require_kaggle_cuda()
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2-v5007',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v8-quality-repair-windowed-rerank',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'adapter_alias_policy': 'sha256-dedupe-v1',
    'source_alias_policy': 'sha256-dedupe-v1',
    'qlora_recovery_schema': 'legalqa-qlora-recovery-v1',
    'qlora_data_schema': 'train-clean-input-v2',
    'inference_pause_schema': 'inference-runtime-pause-v1',
    'offline_wheel_profile': 'hf-stack-2026-08-v1',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-5.14.1-', 'tokenizers-0.22.2-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'safetensors-0.8.0-', 'accelerate-1.14.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentence_transformers-5.7.0-', 'sentencepiece-0.2.1-', 'regex-2026.7.19-', 'click-8.4.2-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_qwen_qlora.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
def source_identity(path):
    digest = hashlib.sha256()
    roots = ('SOURCE_CONTRACT.json', 'pyproject.toml', 'RELEASE_V9_2.md', 'configs', 'kaggle', 'scripts', 'src', 'tests', 'wheelhouse')
    files = []
    for name in roots:
        item = path / name
        files.extend([item] if item.is_file() else ([p for p in item.rglob('*') if p.is_file() and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts and p.suffix != '.pyc'] if item.is_dir() else []))
    for file in sorted(files, key=lambda p: p.relative_to(path).as_posix()):
        digest.update(file.relative_to(path).as_posix().encode('utf-8')); digest.update(b'\0')
        with file.open('rb') as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
source_groups = {}
for path in qualified_sources:
    source_groups.setdefault(source_identity(path), []).append(path)
if len(source_groups) != 1:
    raise RuntimeError(f'Cần đúng một source baseline 0.5007 có train-clean-input-v2; source cũ nhúng trong QLoRA sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
source_aliases = next(iter(source_groups.values()))
selected_source = min(source_aliases, key=lambda p: (len(p.parts), p.as_posix()))
if len(source_aliases) > 1: print('Source aliases share one SHA-256 identity:', source_aliases)
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Môi trường source/offline wheels đã sẵn sàng.')
print('Source:', selected_source)


In [ ]:
resume_input = Path(RESUME_INPUT) if RESUME_INPUT else None
if resume_input is not None and not resume_input.exists():
    raise FileNotFoundError(f'RESUME_INPUT không tồn tại: {resume_input}')
if AUTO_RESUME_TRAINING and resume_input is None:
    print('Auto-resume bật: runner sẽ ưu tiên training-state directory, sau đó qlora_recovery_latest.zip đúng TRAIN_SCOPE.')
command = [
    sys.executable, str(project / 'kaggle' / 'run_qwen_qlora.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_qlora',
    '--train-scope', TRAIN_SCOPE,
    '--dev-size', str(DEV_SIZE),
    '--epochs', str(EPOCHS),
    '--grad-accum', str(GRAD_ACCUM),
    '--max-length', str(MAX_LENGTH),
    '--max-target-tokens', str(MAX_TARGET_TOKENS),
    '--max-evidence-chars', str(MAX_EVIDENCE_CHARS),
    '--lora-r', str(LORA_R),
    '--checkpoint-steps', str(CHECKPOINT_STEPS),
    '--save-total-limit', str(SAVE_TOTAL_LIMIT),
    '--max-runtime-hours', str(MAX_RUNTIME_HOURS),
    '--recovery-output', RECOVERY_OUTPUT,
]
if MAX_TRAIN_SAMPLES is not None:
    command.extend(['--max-train-samples', str(MAX_TRAIN_SAMPLES)])
if resume_input is not None:
    command.extend(['--resume-input', str(resume_input)])
if not AUTO_RESUME_TRAINING:
    command.append('--no-auto-resume')
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path('/kaggle/working/legalqa_qlora')
manifest = root / 'qlora_manifest.json'
assert manifest.exists(), 'Runner chưa tạo manifest; xem log lỗi phía trên.'
payload = json.loads(manifest.read_text(encoding='utf-8'))
adapter = Path(payload['adapter'])
status = payload.get('status')
if status == 'paused':
    recovery = Path(payload['recovery_zip'])
    assert recovery.is_file(), f'Thiếu recovery ZIP: {recovery}'
    assert (adapter / 'training_paused.json').is_file()
    assert payload.get('latest_checkpoint_step') is not None
    print('PAUSED SAFELY: hãy Save Version, tạo Dataset từ qlora_recovery_latest.zip và attach vào session kế tiếp.')
elif status == 'complete':
    assert (adapter / 'adapter_config.json').is_file()
    assert (adapter / 'training_complete.json').is_file()
    assert (root / 'qwen_qlora_adapter.zip').is_file()
else:
    raise RuntimeError(f'Trạng thái QLoRA không hợp lệ: {status!r}')
print(json.dumps(payload, ensure_ascii=False, indent=2))
